In [20]:
from pathlib import Path
import pandas as pd
from narwhals import DataFrame

path_to_csv = Path("../datasets/rnc_dataset_markup.csv")

In [22]:
try:
    df = pd.read_csv(path_to_csv, sep=';', on_bad_lines='skip')
except FileNotFoundError as e:
    print(f"Error: {e}, try to specify path correct.")

In [23]:
df = df.drop(
    columns=["itemId", "split", "targetDetectedMcIds", "targetSplitMcIds", "caseType"]
)

In [24]:
df.tail()



,sourceMcId,sourceMcTitle,description,shouldSplit
2475,101,Ремонт квартир и домов под ключ,Здраствуйте! Облицовка кафелем устройство ГКЛ...,True
2476,101,Ремонт квартир и домов под ключ,Комплексный ремонт квартир \n\nБЕЗ ВЫХОДНЫХ\n\...,False
2477,101,Ремонт квартир и домов под ключ,Ремонт ванных комнат под ключ в Светлогорске. ...,False
2478,101,Ремонт квартир и домов под ключ,"Ремонт квартир, офисов. От штукатурки до плинт...",False
2479,101,Ремонт квартир и домов под ключ,Все види строительство и ремонт\nЦены договорные,False


In [44]:
from utils.description_features_utils import (
    has_bull_markers,
    slash_counter,
    has_slash,
    paragraph_counter,
    word_separately_in_desc,
    count_the_occurrence_of_words_for_separation,
    turkney_count,
)


def features_from_description(df: DataFrame) -> dict:
    desc = df["description"]
    return dict(
        desc_words_count=len(desc.split()),
        desc_length=len(desc),
        has_bull_markers=has_bull_markers(desc),
        has_slashes=has_slash(desc),
        slash_counted=slash_counter(desc),
        paragraph_counted=paragraph_counter(desc),
        turnkey_count=turkney_count(desc),
    )


features_desc = df.apply(features_from_description, axis=1)

df_with_desc_features = pd.concat((df, pd.DataFrame(features_desc.tolist())), axis=1)

ImportError: cannot import name 'l2_norm' from 'utils.description_features_utils' (/Users/dmitrii/catboost_avito/utils/description_features_utils.py)

In [37]:
from numpy._typing import ArrayLike
##### tokenizer & embeddings block
import torch
from torch import nn
from sentence_transformers import SentenceTransformer

bert_model = SentenceTransformer('cointegrated/rubert-tiny2')

enc = bert_model.encode(sentences=["привет, мир", "я на луне"])

print(type(enc).__name__, len(enc))

def row_to_embeds(df_row, /, bert_model_=bert_model) -> ArrayLike:
    title = df_row["sourceMcTitle"]
    desc = df_row["description"]
    sep = "[SEP]"
    sentence = [f"{title + sep + desc}"]
    return bert_model_.encode(sentences=sentence)

import numpy as np

embeds = [row_to_embeds(df.iloc[i], bert_model) for i in range(len(df))]
mtrx = np.vstack(embeds)
df_with_embeds = pd.DataFrame(data=mtrx).add_prefix("embeds")



Loading weights: 100%|██████████| 55/55 [00:00<00:00, 5139.73it/s]
BertModel LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


ndarray 2


In [55]:
from random import randint


def vectors_for_title_and_description(df_row, /, bert_model_=bert_model) -> tuple[ArrayLike, ...]:
    desc, title = df_row["description"], df_row["sourceMcTitle"]
    vectors = bert_model_.encode(sentences=[desc, title])
    return vectors[0], vectors[1]

vectors = vectors_for_title_and_description(df.iloc[randint(1, len(df))])

In [58]:
# L2-Norm as feature block
from numpy import linalg

vectors_from_df = [vectors_for_title_and_description(df.iloc[i]) for i in range(len(df))]



In [27]:
df_final = pd.concat((df_with_desc_features, df_with_embeds), axis=1)

df_final

,sourceMcId,sourceMcTitle,description,shouldSplit,desc_words_count,desc_length,has_bull_markers,has_slashes,slash_counted,paragraph_counted,...,embeds302,embeds303,embeds304,embeds305,embeds306,embeds307,embeds308,embeds309,embeds310,embeds311
0,101,Ремонт квартир и домов под ключ,"Всё виды строительных работ\r\nКачественно, в ...",False,13,85,False,False,0,1,...,0.061496,0.058535,0.019343,0.048706,0.011362,-0.047822,-0.033475,-0.017282,0.071151,-0.023715
1,101,Ремонт квартир и домов под ключ,Профессионально и качественно сделаем ремонт к...,False,107,803,True,False,0,19,...,0.082723,0.043011,0.092585,0.007061,-0.059490,-0.043415,-0.071917,0.053794,0.034265,-0.044075
2,101,Ремонт квартир и домов под ключ,"ремонт квартир, ванной комнате , балкон",False,6,39,False,False,0,0,...,0.080913,0.052698,0.105720,-0.013622,-0.048646,-0.049319,-0.041059,-0.001585,0.036794,-0.034914
3,101,Ремонт квартир и домов под ключ,ЗBОНИТЕ KОHСУЛЬТАЦИЯ БЕCПЛАTНAЯ ПO ТУЛЬСKOЙ ОБ...,True,169,1247,True,True,1,32,...,0.075075,-0.037641,0.001230,0.037845,-0.042515,-0.034750,0.004734,0.057284,0.108178,-0.076892
4,101,Ремонт квартир и домов под ключ,Ремонт квартир любой сложности. Квартиры под к...,False,26,181,False,False,0,0,...,0.072242,0.014644,0.031661,0.029496,-0.020382,-0.030702,0.009245,0.003996,0.094669,-0.075401
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2475,101,Ремонт квартир и домов под ключ,Здраствуйте! Облицовка кафелем устройство ГКЛ...,True,15,152,False,False,0,1,...,0.067344,0.036425,0.087353,-0.009215,-0.014532,-0.005530,-0.063023,0.027905,0.047493,-0.067158
2476,101,Ремонт квартир и домов под ключ,Комплексный ремонт квартир \n\nБЕЗ ВЫХОДНЫХ\n\...,False,11,84,False,False,0,6,...,0.061211,0.045668,0.044869,0.048461,-0.057116,-0.026143,-0.005377,0.013173,0.074711,-0.049196
2477,101,Ремонт квартир и домов под ключ,Ремонт ванных комнат под ключ в Светлогорске. ...,False,13,80,False,False,0,0,...,0.045360,0.073630,0.106563,0.034882,-0.020405,-0.006367,-0.050745,0.015481,0.070522,-0.054931
2478,101,Ремонт квартир и домов под ключ,"Ремонт квартир, офисов. От штукатурки до плинт...",False,7,53,False,False,0,0,...,0.069940,0.095810,0.075583,-0.005570,-0.026040,-0.021695,-0.036841,-0.012547,0.063319,-0.054734


In [28]:
print(
    f"Random should split string for repr: \n\n {df_with_desc_features.iloc[1].to_string()}"
)

Random should split string for repr: 

 sourceMcId                                                         101
sourceMcTitle                          Ремонт квартир и домов под ключ
description          Профессионально и качественно сделаем ремонт к...
shouldSplit                                                      False
desc_words_count                                                   107
desc_length                                                        803
has_bull_markers                                                  True
has_slashes                                                      False
slash_counted                                                        0
paragraph_counted                                                   19
turnkey_count                                                        1


In [29]:
from sklearn.model_selection import train_test_split

X = df_final.drop(columns=["shouldSplit", "sourceMcTitle", "description"])
y = df_final["shouldSplit"]

X_train, X_holdout, y_train, y_holdout = train_test_split(
    X, y, test_size=0.15, random_state=45, shuffle=True, stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_holdout,
    y_holdout,
    test_size=0.5,
    random_state=52,
    shuffle=True,
    stratify=y_holdout,
)


In [30]:
from catboost import CatBoostClassifier

model = CatBoostClassifier(
    iterations=3000,
    learning_rate=0.03,
    loss_function="Logloss",
    eval_metric="AUC",
    auto_class_weights="Balanced",
)

model.fit(
    X_train,
    y_train,
    eval_set=(X_val, y_val),
    early_stopping_rounds=50,
    use_best_model=True,
    cat_features=["sourceMcId"],
    verbose=50,)


0:	test: 0.6519395	best: 0.6519395 (0)	total: 38.1ms	remaining: 1m 54s
50:	test: 0.7924314	best: 0.8020814 (46)	total: 1.54s	remaining: 1m 29s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.8020813623
bestIteration = 46

Shrink model to first 47 iterations.


CatBoostClassifier(auto_class_weights='Balanced', eval_metric='AUC', iterations=3000, learning_rate=0.03, loss_function='Logloss')

In [31]:
from sklearn.metrics import accuracy_score, confusion_matrix

y_pred = model.predict(X_val)

acc_score = accuracy_score(y_val, y_pred)
cfs_mtrx = confusion_matrix(y_val, y_pred)

In [32]:
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, log_loss, classification_report


def report_binary(model, X, y, name="data"):
      y = np.asarray(y).astype(int)
      y_pred = model.predict(X).astype(int)
      proba = model.predict_proba(X)[:, 1]
      print(f"=== {name} ===")
      print(f"accuracy:  {accuracy_score(y, y_pred):.4f}")
      print(f"precision: {precision_score(y, y_pred, zero_division=0):.4f}")
      print(f"recall:    {recall_score(y, y_pred, zero_division=0):.4f}")
      print(f"f1:        {f1_score(y, y_pred, zero_division=0):.4f}")
      print(f"roc_auc:   {roc_auc_score(y, proba):.4f}")
      print(f"log_loss:  {log_loss(y, proba):.4f}")
      cm = confusion_matrix(y, y_pred, labels=[0, 1])
      print("confusion_matrix [rows=true 0,1 | cols=pred 0,1]:")
      print(cm)
      print(classification_report(y, y_pred, labels=[0, 1], target_names=["no_split", "split"]))

In [33]:
print(f"Report for validation set: ---------\n\n{report_binary(model, X_val, y_val, name="validation")}")
print(f"Report for validation set: ---------\n\n{report_binary(model, X_test, y_test, name="validation")}")



=== validation ===
accuracy:  0.7581
precision: 0.4194
recall:    0.7429
f1:        0.5361
roc_auc:   0.8021
log_loss:  0.5749
confusion_matrix [rows=true 0,1 | cols=pred 0,1]:
[[115  36]
 [  9  26]]
              precision    recall  f1-score   support

    no_split       0.93      0.76      0.84       151
       split       0.42      0.74      0.54        35

    accuracy                           0.76       186
   macro avg       0.67      0.75      0.69       186
weighted avg       0.83      0.76      0.78       186

Report for validation set: ---------

None
=== validation ===
accuracy:  0.6828
precision: 0.3333
recall:    0.6389
f1:        0.4381
roc_auc:   0.7789
log_loss:  0.5812
confusion_matrix [rows=true 0,1 | cols=pred 0,1]:
[[104  46]
 [ 13  23]]
              precision    recall  f1-score   support

    no_split       0.89      0.69      0.78       150
       split       0.33      0.64      0.44        36

    accuracy                           0.68       186
   macro avg

In [34]:
def predict(source_mc_id: int, title: str, description: str, bert_model):
    row = {
    "sourceMcId": source_mc_id,
    "desc_words_count": len(description.split()),
    "desc_length": len(description),
    "has_bull_markers": has_bull_markers(description),
    "has_slashes": has_slash(description),
    "slash_counted": slash_counter(description),
    "paragraph_counted": paragraph_counter(description),
    "turnkey_count": turkney_count(description),
}
    sep = "[SEP]"
    sentence = [f"{title + sep + description}"]
    embeddings = bert_model.encode(sentences=sentence)
    arr = np.vstack(embeddings)

    pred_df_with_embeds = pd.DataFrame(data=arr).add_prefix("embeds")
    final_to_pred = pd.concat((pd.DataFrame([row]), pred_df_with_embeds), axis=1)
    return model.predict_proba(final_to_pred)





In [18]:
import pandas as pd
from catboost import Pool

description = "Ремонт любой сложности!!! / замена фреона / ремонт компрессора / замена термостата / выезд на дом / гарантия 12 месяцев / чек / запчасти в наличии."
source_mc_title = "Ремонт квартир и домов под ключ"

predict(title=source_mc_title, description=description, bert_model=bert_model, source_mc_id=101)



array([[0.76853747, 0.23146253]])

In [ ]:
print("=== Колонки X_train ===")
print(f"Всего: {len(X_train.columns)}")
print(f"Первые 15: {list(X_train.columns[:15])}")
print(f"Последние 5: {list(X_train.columns[-5:])}")



In [35]:
# Тест 1: Максимальный семантический разрыв + Структурные триггеры
# (Тут и разный смысл, и много слов, и слэши, и слово-триггер "отдельно")
title_1 = "Ремонт холодильников"
description_1 = (
    "Ремонт холодильников любой сложности на дому. / "
    "Отдельно предлагаю услуги по дрессировке собак и выгулу домашних животных. / "
    "Также занимаюсь продажей запчастей для тракторов."
)

# Тест 2: Твой кейс с напольными покрытиями, но "дожатый" по объему
# (Если короткий вариант не сработал, этот должен из-за веса фичи desc_length)
title_2 = "Напольные покрытия"
description_2 = (
    "Укладка ламината, кварцвинила и паркетной доски быстро и качественно. "
    "Также выполняем полный монтаж перегородок из гипсокартона, возведение стен и перепланировку. "
    "Отдельно возможна установка натяжных потолков во всей квартире. Звоните!"
)

# Тест 3: Чистый "Винегрет" (много слэшей и разных ниш)
title_3 = "Услуги сантехника"
description_3 = "Установка крана / Ремонт стиральных машин / Поклейка обоев / Перевозка мебели / Грузчики"

# Функция для запуска тестов
def run_tests():
    for i, (t, d) in enumerate([(title_1, description_1), (title_2, description_2), (title_3, description_3)], 1):
        # source_mc_id берем из твоей категории (например, 101 или 109)
        res = predict(title=t, description=d, bert_model=bert_model, source_mc_id=101)
        print(f"Тест {i}: [No Split, Split] -> {res}")

run_tests()



Тест 1: [No Split, Split] -> [[0.65988039 0.34011961]]
Тест 2: [No Split, Split] -> [[0.42415261 0.57584739]]
Тест 3: [No Split, Split] -> [[0.57071159 0.42928841]]


In [36]:
import matplotlib.pyplot as plt
fi = model.get_feature_importance(prettified=True)
print(fi.head(10))

  Feature Id  Importances
0  embeds155     4.472534
1  embeds221     3.948920
2  embeds260     3.668878
3  embeds239     2.997098
4   embeds42     2.932962
5   embeds28     2.714824
6  embeds192     2.693639
7   embeds71     2.313650
8  embeds230     2.138790
9  embeds159     2.115937
